# Développez un assistant pour la recommandation d'évènements culturels

## Contexte

Vous êtes ingénieur data freelance spécialisé en intégration de modèles NLP et gestion de bases de données vectorielles. Vous travaillez pour l’entreprise Puls-Events, une société innovante dans le domaine de la gestion d’événements culturels.

 

L’entreprise Puls-Events propose une plateforme web permettant aux utilisateurs de découvrir et de suivre des événements culturels en temps réel. Elle collecte des informations à partir de diverses sources, comme Open Agenda, pour proposer des événements adaptés aux préférences de ses utilisateurs, filtrables par lieu et par période.

 

Elle souhaite tester l’intégration d’un chatbot intelligent, capable de fournir des recommandations personnalisées et augmentées par des réponses précises issues de données d’événements. 

 

Ce chatbot, soutenu par un système de génération de réponses augmentées (RAG), doit être capable d'extraire et d'analyser des informations à partir d'une base de données vectorielle.

 

Vous êtes chargé de réaliser un premier Proof of Concept (POC) pour démontrer la faisabilité de ce concept. 

 

Vous recevez un message de votre responsable technique : 

 

De  : Jérémy

A : moi

Objet :  version fonctionnelle du POC RAG

Salut,

 

Nous avons besoin que tu termines une version fonctionnelle du POC RAG. Il sera présenté en réunion avec les équipes produit et marketing pour vérifier l’intérêt du projet et envisager son déploiement à plus grande échelle.

 

Comme discuté ensemble, nous allons nous appuyer sur la source de données des événements publics Open Agenda. Tu pourras te limiter au périmètre géographique de ton choix. Veille cependant à ce qu’il y ait uniquement des événements récents de moins d’un an.

 

Ce qui est attendu :

* **Environnement virtuel** : Pour que le projet soit facilement reproductible, dépendances et documentation sous forme d’un readme.
* **Système RAG fonctionnel** : Code complet du système RAG versionné dans un repo git, avec intégration de LangChain, Mistral et Faiss. Le système doit être en mesure de reconstruire la base vectorielle sur demande (scripts de pré-processing et de vectorisation des données).
* **Rapport technique (PDF ou Word)** : Document entre 5 et 10 pages, expliquant l’architecture du système, tes choix techniques, les modèles sélectionnés, les résultats obtenus pour le POC et les recommandations pour la version finale.
* **Tests unitaires (Python)** : Prévois un fichier .py pour tester que les données intégrées dans la base vectorielle correspondent bien à des événements de moins d’un an dans la région géographique que tu auras sélectionnée.
* **Présentation (PowerPoint)** : Entre 10 et 15 slides pour exposer le concept du projet, les résultats du POC.
Démo live du système : Prépare une démo live pour illustrer les capacités du POC.
N’hésite pas si tu as besoin de précisions sur le périmètre ou les attentes. On compte sur toi pour que tout soit prêt dans les délais !

 

Ah oui, j’oubliais... Pour évaluer le système, nous aurons besoin de créer un jeu de données test annoté de questions/réponses. Ce jeu test sera utilisé pour mesurer la qualité des réponses par rapport aux réponses annotées.

 

Merci et bon courage !

 

Jérémy

## Étape 1 - Préparez l'environnement de travail



Création de l'environnement virtuel avec la commande suivante:

```
(base) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>python -m venv .venv_rag
```
 Désactivation de l'environnement de base puis activation de l'environnement virtuel `venv_rag`

 ```
(base) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>conda deactivate

C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>.venv_rag\Scripts\activate

(.venv_rag) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>

```
Installation des packages nécessaires:

```
(.venv_rag) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>pip install langchain

(.venv_rag) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>pip install faiss-cpu

(.venv_rag) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>pip install mistralai


```
https://github.com/mistralai/client-python



In [1]:
import sys
print(sys.executable)

C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11\.venv_rag\Scripts\python.exe


### Vérification de l'installation des packages
Le script suivant est exécuté dans l'invite de commande avec la commande `python script_verif_install_packages.py`:

In [30]:
import importlib.util
import subprocess
import sys

def package_verification(module, pip_name): 
    """
    Vérifie si un package est installé et affiche ses informations.
    """
    if importlib.util.find_spec(module): # importlib.util.find_spec() retourne None si le module n'est pas trouvé
        # Le package est installé : afficher ses infos
        print("Le package est bien installé, voici ses informations:\n")
        subprocess.run([sys.executable, "-m", "pip", "show", pip_name]) # subprocess.run() exécute la commande pip show pour afficher les informations du package
    else:
        print(f"Le package {pip_name} n'est pas installé. Taper: pip install {pip_name} pour l'installer.")

if __name__ == "__main__":
    packages = {
    "langchain": "langchain",
    "faiss": "faiss-cpu",
    "mistralai": "mistralai",    
}
    
    for module, pip_name in packages.items():
        print(f"==============Vérification du package {pip_name}==============\n")
        package_verification(module, pip_name)
        print()

==============Vérification du package langchain==============

Le package est bien installé, voici ses informations:


==============Vérification du package faiss-cpu==============

Le package est bien installé, voici ses informations:


==============Vérification du package mistralai==============

Le package est bien installé, voici ses informations:




Résultat de la commande `python script_verif_install_packages.py`:

```
(.venv_rag) C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11>python script_verif_install_packages.py
==============Vérification du package langchain==============

Le package est bien installé, voici ses informations:

Name: langchain
Version: 1.4.3
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author:
Author-email:
License: MIT
Location: C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11\.venv_rag\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by:

==============Vérification du package faiss-cpu==============

Le package est bien installé, voici ses informations:

Name: faiss-cpu
Version: 1.15.1
Summary: A library for efficient similarity search and clustering of dense vectors.
Home-page: https://github.com/facebookresearch/faiss
Author: Meta AI Research
Author-email:
License-Expression: MIT
Location: C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11\.venv_rag\Lib\site-packages
Requires: numpy, packaging
Required-by:

==============Vérification du package mistralai==============

Le package est bien installé, voici ses informations:

Name: mistralai
Version: 3.0.0
Summary: Python Client SDK for the Mistral AI API.
Home-page:
Author: Mistral
Author-email:
License:
Location: C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11\.venv_rag\Lib\site-packages
Requires: eval-type-backport, httpx2, jsonpath-python, opentelemetry-api, opentelemetry-semantic-conventions, pydantic, python-dateutil, typing-inspection
Required-by:

==============Vérification du package sentence-transformers==============

Le package est bien installé, voici ses informations:

Name: sentence-transformers
Version: 6.1.0
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author:
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License-Expression: Apache-2.0
Location: C:\Users\sebas\Documents\cours_openclassrooms\data_engineer\projet_11\.venv_rag\Lib\site-packages
Requires: huggingface-hub, numpy, scikit-learn, scipy, tokenizers, torch, tqdm, transformers, typing_extensions
Required-by:
```

### Focus sur FAISS
Faiss-cpu est la version pour processeur (CPU) de FAISS (Facebook AI Similarity Search), une bibliothèque open source développée par Meta pour la recherche rapide de similarité et le regroupement (clustering) de grands ensembles de vecteurs.

Afin de vérifier la bonne installation de faiss, suivre la procédure présentée sous Github: https://github.com/facebookresearch/faiss/wiki/Getting-started

In [21]:
# Vérifier que FAISS est correctement installé et que la recherche CPU fonctionne
import numpy as np

print("\n=== Test fonctionnel de FAISS (CPU) ===")

# Création de données fictives (100000 vecteurs de dimension 64, 10000 questions)
d = 64
nb = 100000
nq = 10000
np.random.seed(1234) # rendre reproductible
xb = np.random.random((nb, d)).astype("float32") 
xb[:, 0] += np.arange(nb) / 1000.
# xb représente la base de données contenant l’ensemble des vecteurs à indexer et dans laquelle nous effectuerons les recherches. Elle est de dimension nb × d.

xq = np.random.random((nq, d)).astype("float32")
xq[:, 0] += np.arange(nq) / 1000.
# xq désigne les vecteurs de requête pour lesquels nous devons trouver les plus proches voisins. Sa dimension est nq × d. 
# Si nous avons un seul vecteur de requête, alors nq = 1
    
# Initialiser un index CPU classique (IndexFlatL2)
index = faiss.IndexFlatL2(d) # IndexFlatL2 est un index FAISS qui s'exécute sur CPU par défaut.
print(f"Index est-il entraîné? {index.is_trained}")

# Ajouter les vecteurs à l'index
index.add(xb)
print(f"Nombre total de vecteurs indexés : {index.ntotal}")

# Effectuer une recherche de test
k=2 # voir les 2 voisins les plus proches
D, I = index.search(xb[:5], k) # Test de cohérence
print(I) # doit commencer par 0, correspond aux indices, classés du plus proche au plus éloigné
print(D) # contient les distances de chaque vecteur
D, I = index.search(xq, k) 
print("Voisins des 5 premiers vecteurs:\n", I[:5])   
print("Voisin des 5 derniers vecteurs:\n", I[-5:])
print("FAISS fonctionne correctement")


=== Test fonctionnel de FAISS (CPU) ===
Index est-il entraîné? True
Nombre total de vecteurs indexés : 100000
[[  0 393]
 [  1 555]
 [  2 304]
 [  3 173]
 [  4 288]]
[[1.1444092e-05 7.1751785e+00]
 [0.0000000e+00 6.3235626e+00]
 [7.6293945e-06 5.7964058e+00]
 [3.8146973e-06 7.2779045e+00]
 [0.0000000e+00 6.7638016e+00]]
Voisins des 5 premiers vecteurs:
 [[381 207]
 [526 911]
 [838 527]
 [196 184]
 [526 377]]
Voisin des 5 derniers vecteurs:
 [[ 9900 10500]
 [11055 10895]
 [11353 11103]
 [10571 10664]
 [ 9628  9554]]
FAISS fonctionne correctement


## Étape 2 - Effectuez le préprocessing des données Open Agenda

### Récupération des données
Les données sont présentes sur la plateforme publique https://public.opendatasoft.com/explore/assets/evenements-publics-openagenda/

![Opendatasoft_opendataagenda.png](Opendatasoft_opendataagenda.png)


Dans la fenêtre Requête, ^les champs suivants sont complétés:


SELECT

uid, slug, title_fr, description_fr, longdescription_fr, conditions_fr, keywords_fr, lastdate_end,  location_coordinates, location_name, location_address,  location_district, location_postalcode, location_city, location_department,  location_region, location_countrycode,  country_fr

WHERE

location_city = "Paris" AND lastdate_end>="2025-10-05T00:00:00+00:00"

LANGUAGE

fr


Se rendre sur le site https://public.opendatasoft.com/api-console/explore/v2.1/catalog/datasets/evenements-publics-openagenda/ en allant dans la console API et en cliquant sur le bouton {...} pour API(Swagger)

Compléter les champs de  `GET/catalog/dataset/evenements-publics-openagenda/exports/{format}`

Compléter les paramètres ainsi:
* **format:** json
* **select:** uid, slug, title_fr, description_fr, longdescription_fr, conditions_fr, keywords_fr, lastdate_end,  location_coordinates, location_name, location_address,  location_district, location_postalcode, location_city, location_department,  location_region, location_countrycode,  country_fr
* **where:** location_city = "Paris" AND lastdate_end>="2025-10-05T00:00:00+00:00"
* **limit:** -1 (laisser -1 pour récupérer tous les enregistrements)
* **lang:** fr (pour français)

Récupérer la `Request URL` générée:
https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/exports/json?select=uid%2C%20slug%2C%20title_fr%2C%20description_fr%2C%20longdescription_fr%2C%20conditions_fr%2C%20keywords_fr%2C%20lastdate_end%2C%20%20location_coordinates%2C%20location_name%2C%20location_address%2C%20%20location_district%2C%20location_postalcode%2C%20location_city%2C%20location_department%2C%20%20location_region%2C%20location_countrycode%2C%20%20country_fr&where=location_city%20%3D%20%22Paris%22%20AND%20lastdate_end%3E%3D%222025-10-05T00%3A00%3A00%2B00%3A00%22&limit=-1&lang=fr&timezone=UTC&use_labels=false&compressed=false&epsg=4326

In [36]:
import requests
url = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/exports/json?select=uid%2C%20slug%2C%20title_fr%2C%20description_fr%2C%20longdescription_fr%2C%20conditions_fr%2C%20keywords_fr%2C%20lastdate_end%2C%20%20location_coordinates%2C%20location_name%2C%20location_address%2C%20%20location_district%2C%20location_postalcode%2C%20location_city%2C%20location_department%2C%20%20location_region%2C%20location_countrycode%2C%20%20country_fr&where=location_city%20%3D%20%22Paris%22%20AND%20lastdate_end%3E%3D%222025-10-05T00%3A00%3A00%2B00%3A00%22&limit=-1&lang=fr&timezone=UTC&use_labels=false&compressed=false&epsg=4326"
response = requests.get(url)

In [37]:
import pandas as pd
df = pd.DataFrame(response.json())

In [39]:
df.shape

(9541, 18)

In [41]:
df.head(2)

,uid,slug,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,lastdate_end,location_coordinates,location_name,location_address,location_district,location_postalcode,location_city,location_department,location_region,location_countrycode,country_fr
0,97781943,decouverte-nature-en-famille-avec-enfants-de-2...,Dimanche au Vert en famille avec enfants de 3 ...,Balade Nature & Montessori. (Re)connectez-vous...,<p>Les Dimanches au Vert en famille proposent ...,Tarifs : Un duo 1 enfant /1 parent 35€ ; Un en...,"[Nature, Sortie nature, enfant (éveil nature…)]",2026-06-07T12:00:00+00:00,"{'lon': 2.382098, 'lat': 48.879555}","Parc des Buttes-Chaumont, 75019 Paris, France","Parc des Buttes-Chaumont, 75019 Paris, France",Paris,75019,Paris,Paris,Île-de-France,FR,France (Métropole)
1,99134796,la-vie-sauvage-secrete-du-bois-de-vincennes-au...,La vie sauvage secrète du Bois de Vincennes au...,Une soirée insolite pour partir à la découvert...,<p>La gestion écologique du Bois de Vincennes ...,Tarifs : Adulte 13€ ; Enfant 8 à 12 ans 9€,"[Nature, Sortie nature, biodiversité, oiseaux,...",2026-06-17T23:30:00+00:00,"{'lon': 2.435365, 'lat': 48.839415}","Esp. Saint-Louis, 75012 Paris, France","Esp. Saint-Louis, 75012 Paris, France",Quartier de Picpus,75012,Paris,Département de Paris,IDF,FR,France (Métropole)


In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9541 entries, 0 to 9540
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   uid                   9541 non-null   str   
 1   slug                  9541 non-null   str   
 2   title_fr              9504 non-null   str   
 3   description_fr        9504 non-null   str   
 4   longdescription_fr    9017 non-null   str   
 5   conditions_fr         4488 non-null   str   
 6   keywords_fr           6116 non-null   object
 7   lastdate_end          9541 non-null   str   
 8   location_coordinates  9541 non-null   object
 9   location_name         9541 non-null   str   
 10  location_address      9541 non-null   str   
 11  location_district     9232 non-null   str   
 12  location_postalcode   9448 non-null   str   
 13  location_city         9541 non-null   str   
 14  location_department   9343 non-null   str   
 15  location_region       9534 non-null   str   
 16 

### Nettoyage des données

#### Dédoublement

In [52]:
df.duplicated(subset="uid").sum()

np.int64(0)

In [53]:
df1 = df.drop_duplicates(subset="uid",ignore_index=True)

In [46]:
df1.shape

(9541, 18)

#### Suppression des descriptions vides

In [59]:
df1["description_fr"].isna().sum()

np.int64(37)

In [69]:
df2 = df1[~df1["description_fr"].isna()].reset_index(drop=True)

In [71]:
df2.head()

,uid,slug,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,lastdate_end,location_coordinates,location_name,location_address,location_district,location_postalcode,location_city,location_department,location_region,location_countrycode,country_fr
0,97781943,decouverte-nature-en-famille-avec-enfants-de-2...,Dimanche au Vert en famille avec enfants de 3 ...,Balade Nature & Montessori. (Re)connectez-vous...,<p>Les Dimanches au Vert en famille proposent ...,Tarifs : Un duo 1 enfant /1 parent 35€ ; Un en...,"[Nature, Sortie nature, enfant (éveil nature…)]",2026-06-07T12:00:00+00:00,"{'lon': 2.382098, 'lat': 48.879555}","Parc des Buttes-Chaumont, 75019 Paris, France","Parc des Buttes-Chaumont, 75019 Paris, France",Paris,75019,Paris,Paris,Île-de-France,FR,France (Métropole)
1,99134796,la-vie-sauvage-secrete-du-bois-de-vincennes-au...,La vie sauvage secrète du Bois de Vincennes au...,Une soirée insolite pour partir à la découvert...,<p>La gestion écologique du Bois de Vincennes ...,Tarifs : Adulte 13€ ; Enfant 8 à 12 ans 9€,"[Nature, Sortie nature, biodiversité, oiseaux,...",2026-06-17T23:30:00+00:00,"{'lon': 2.435365, 'lat': 48.839415}","Esp. Saint-Louis, 75012 Paris, France","Esp. Saint-Louis, 75012 Paris, France",Quartier de Picpus,75012,Paris,Département de Paris,IDF,FR,France (Métropole)
2,91813967,recrutement-aide-soignant-e-hopital-sainte-perine,Recrutement Aide Soignant (e)- Hôpital Sainte-...,L'HOPITAL SAINTE-PERINE ( Groupe AP-HP) recrut...,<p>L'HOPITAL SAINTE-PERINE ( Groupe AP-HP) rec...,NaN,"[1 jeune 1 solution, Recrutement]",2025-12-18T12:00:00+00:00,"{'lon': 2.266908, 'lat': 48.84507}",Paris - Agence Rueil-Malmaison,75016 Paris,Quartier d'Auteuil,75016,Paris,Paris,Ile-de-France,FR,France (Métropole)
3,22683319,fiap-jean-monnet,FIAP Jean Monnet,"1 ERP de type O avec activité L, N, P et R, 1 ...",<p>Le FIAP Paris est un Centre International d...,NaN,None,2032-01-01T03:00:00+00:00,"{'lon': 2.33821, 'lat': 48.830538}",Fiap jean Monnet,30 rue Cabanis 75014 Paris,Quartier du Parc-de-Montsouris,75014,Paris,Paris,Île-de-France,FR,France (Métropole)
4,35846321,linde-nouvel-eldorado-de-la-beaute,"L'Inde, nouvel eldorado de la beauté ?","L'Inde, nouvel eldorado de la beauté ? Par HEC...","<p>Cher(e) camarade, </p>\n<p>Le marché indie...",sur inscription,[Inde nouvel eldorado beauté HEC Alumni Paris],2026-01-14T21:00:00+00:00,"{'lon': 2.310012, 'lat': 48.865877}",HEC Alumni,9 avenue Franklin Roosevelt 75008 Paris,Quartier des Champs-Élysées,75008,Paris,Paris,Île-de-France,FR,France (Métropole)


In [83]:
df2.shape

(9504, 18)

In [85]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 9504 entries, 0 to 9503
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   uid                   9504 non-null   str   
 1   slug                  9504 non-null   str   
 2   title_fr              9504 non-null   str   
 3   description_fr        9504 non-null   str   
 4   longdescription_fr    9017 non-null   str   
 5   conditions_fr         4488 non-null   str   
 6   keywords_fr           6116 non-null   object
 7   lastdate_end          9504 non-null   str   
 8   location_coordinates  9504 non-null   object
 9   location_name         9504 non-null   str   
 10  location_address      9504 non-null   str   
 11  location_district     9196 non-null   str   
 12  location_postalcode   9412 non-null   str   
 13  location_city         9504 non-null   str   
 14  location_department   9307 non-null   str   
 15  location_region       9497 non-null   str   
 16 

#### Suppression des descriptions longues nulles


In [88]:
df2["longdescription_fr"].isna().sum()

np.int64(487)

In [91]:
df3 = df2[~df2["longdescription_fr"].isna()].reset_index(drop=True)

In [92]:
df3.shape

(9017, 18)

In [93]:
df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 9017 entries, 0 to 9016
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   uid                   9017 non-null   str   
 1   slug                  9017 non-null   str   
 2   title_fr              9017 non-null   str   
 3   description_fr        9017 non-null   str   
 4   longdescription_fr    9017 non-null   str   
 5   conditions_fr         4337 non-null   str   
 6   keywords_fr           5792 non-null   object
 7   lastdate_end          9017 non-null   str   
 8   location_coordinates  9017 non-null   object
 9   location_name         9017 non-null   str   
 10  location_address      9017 non-null   str   
 11  location_district     8713 non-null   str   
 12  location_postalcode   8926 non-null   str   
 13  location_city         9017 non-null   str   
 14  location_department   8822 non-null   str   
 15  location_region       9010 non-null   str   
 16 

#### Suppression des balises html

In [75]:
print(df2["longdescription_fr"][0])

<p>Les Dimanches au Vert en famille proposent aux parents et leurs enfants de 3 à 10 ans des expériences Nature pour leur permettre de vivre ensemble des aventures riches et complices au cœur de 2 lieux parisiens ou proche de Paris les dimanches de 10 h à 12h.<br>Chaque Saison à travers nos 5 sens.  </p>
<p> <strong>Au programme :</strong> <br>- Aventure et Apprentissages avec les secrets de Dame Nature distillés au gré des rencontres végétales ou animales, en fonction du moment et des besoins de chacun. Éveiller les curiosités, entretenir ( ou réveiller ;) l’émerveillement, créer l'envie d’aller plus loin dans l’exploration.<br>- Moment questions Nature et une pause Yoga Nature.<br>- Jeux et contes sur la Nature. Une façon ludique, sensorielle et expérimentale de découvrir les comportements de la faune et la flore à chaque saison. Et pouvant donner lieu à de grands moments de fous rires.<br>- Des Ateliers Créatifs 100% Nature à 4 mains. Petits et grands se font plaisir à fabriquer ens

les descriptions longues sont écrites en HTML. Il est nécessaire de retirer les balises afin de permettre au LLM de saisir les informations correctement.

In [76]:
from bs4 import BeautifulSoup

In [97]:
df3["bs_longdescription_fr"] = df3["longdescription_fr"].apply(lambda x : BeautifulSoup(x,'html.parser').get_text())

In [99]:
print(df3["bs_longdescription_fr"][0])

Les Dimanches au Vert en famille proposent aux parents et leurs enfants de 3 à 10 ans des expériences Nature pour leur permettre de vivre ensemble des aventures riches et complices au cœur de 2 lieux parisiens ou proche de Paris les dimanches de 10 h à 12h.Chaque Saison à travers nos 5 sens.  
 Au programme : - Aventure et Apprentissages avec les secrets de Dame Nature distillés au gré des rencontres végétales ou animales, en fonction du moment et des besoins de chacun. Éveiller les curiosités, entretenir ( ou réveiller ;) l’émerveillement, créer l'envie d’aller plus loin dans l’exploration.- Moment questions Nature et une pause Yoga Nature.- Jeux et contes sur la Nature. Une façon ludique, sensorielle et expérimentale de découvrir les comportements de la faune et la flore à chaque saison. Et pouvant donner lieu à de grands moments de fous rires.- Des Ateliers Créatifs 100% Nature à 4 mains. Petits et grands se font plaisir à fabriquer ensemble, toujours en fonction de la saison, des c

In [32]:
dates au format date

<Response [401]>

In [ ]:
coordonnées